# 01 · The `.dly` format: parse one station by hand, then check Spark agrees

**Question:** what does a GHCN-Daily file actually contain, and can the Spark parser be trusted?

GHCN-Daily stores one text file per station. Each line is one *station, year, month, element*, with up to 31 daily values packed into fixed-width slots. Before parsing millions of lines in Spark, parse one station the slow, obvious way in pandas, then require Spark to produce identical rows.

In [1]:
import pathlib
import pandas as pd
from ghcnd_quality.spark import get_spark
from ghcnd_quality.dly import read_dly, CORE_ELEMENTS, QFLAG_MEANING

pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)
RAW = pathlib.Path("../data/raw")

## Pick a station

The station with the longest TMAX record among the downloaded files (using the inventory file, itself fixed-width):

In [2]:
inv = pd.read_fwf(RAW / "ghcnd-inventory.txt", colspecs=[(0, 11), (31, 35), (36, 40), (41, 45)],
                  names=["station_id", "element", "first_year", "last_year"])
have = {p.stem for p in (RAW / "ghcnd_all").glob("*.dly")}
cand = inv[(inv.element == "TMAX") & inv.station_id.isin(have)].assign(span=lambda d: d.last_year - d.first_year)
STATION = cand.sort_values(["span", "station_id"], ascending=[False, True]).station_id.iloc[0]
PATH = RAW / "ghcnd_all" / f"{STATION}.dly"
print(STATION, cand[cand.station_id == STATION][["first_year", "last_year"]].iloc[0].tolist())

UK000047811 [1844, 2026]


## Anatomy of one line

A line with real quality flags in it (or the first TMAX line if the station has none):

In [3]:
lines = PATH.read_text().splitlines()
flagged = [l for l in lines if l[17:21] == "TMAX" and any(l[21 + d * 8 + 6] != " " for d in range(31))]
line = (flagged or [l for l in lines if l[17:21] == "TMAX"])[0]

print(f"station : {line[0:11]}\nyear    : {line[11:15]}\nmonth   : {line[15:17]}\nelement : {line[17:21]}\n")
print("slot  value  m q s")
for day in range(1, 8):
    s = line[21 + (day - 1) * 8: 21 + day * 8]
    print(f"{day:>3}   {s[:5]}  {s[5]!r} {s[6]!r} {s[7]!r}")

station : UK000047811
year    : 1850
month   : 11
element : TMAX

slot  value  m q s
  1     154  ' ' ' ' 'E'
  2     131  ' ' ' ' 'E'
  3     131  ' ' ' ' 'E'
  4     120  ' ' ' ' 'E'
  5     128  ' ' ' ' 'E'
  6     122  ' ' ' ' 'E'
  7     120  ' ' ' ' 'E'


Each 8-character slot is `value (5)`, `measurement flag (1)`, `quality flag (1)`, `source flag (1)`. A value of `-9999` means missing. A month has at most 31 slots, so short months are padded with `-9999`.

## The slow, obvious parse (pandas)

In [4]:
def parse_dly_pandas(path):
    rows = []
    for line in open(path):
        station, year, month, element = line[0:11], int(line[11:15]), int(line[15:17]), line[17:21]
        if element not in CORE_ELEMENTS:
            continue
        for day in range(1, 32):
            off = 21 + (day - 1) * 8
            value = int(line[off:off + 5])
            if value == -9999:
                continue
            try:
                date = pd.Timestamp(year, month, day)
            except ValueError:            # e.g. day 31 of a 30-day month
                continue
            rows.append((station, date, element, value,
                         line[off + 5].strip() or None, line[off + 6].strip() or None, line[off + 7].strip() or None))
    return pd.DataFrame(rows, columns=["station_id", "date", "element", "value", "mflag", "qflag", "sflag"])

py = parse_dly_pandas(PATH)
print(f"{len(py):,} observed daily values")
py.groupby("element").size().rename("observations")

202,097 observed daily values


element
PRCP    68732
TMAX    66681
TMIN    66684
Name: observations, dtype: int64

## Spark must agree, row for row

In [5]:
spark = get_spark("nb01")
sp = read_dly(spark, str(PATH)).toPandas()
sp["date"] = pd.to_datetime(sp["date"])

key = ["station_id", "date", "element"]
merged = py.merge(sp, on=key, how="outer", suffixes=("_py", "_spark"), indicator=True)
print(merged["_merge"].value_counts().to_string())
assert (merged["_merge"] == "both").all(), "row sets differ"
for col in ("value", "mflag", "qflag", "sflag"):
    a, b = merged[f"{col}_py"], merged[f"{col}_spark"]
    assert ((a == b) | (a.isna() & b.isna())).all(), f"{col} differs"
print("identical on", len(merged), "rows: value, mflag, qflag, sflag")

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=40689 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=40689 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/10 18:29:18 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/10/10 18:29:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/10 18:29:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/10/10 18:29:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


_merge
both          202097
left_only          0
right_only         0
identical on 202097 rows: value, mflag, qflag, sflag


## How much is missing?

Counting every slot a line could hold, including the padding for short months:

In [6]:
total_slots = missing = 0
for l in lines:
    if l[17:21] not in CORE_ELEMENTS:
        continue
    for day in range(31):
        total_slots += 1
        missing += l[21 + day * 8: 26 + day * 8] == "-9999"
print(f"{missing / total_slots:.1%} of the {total_slots:,} slots in core-element lines are -9999 (missing or padding)")

1.9% of the 206,088 slots in core-element lines are -9999 (missing or padding)


## Quality flags

NOAA's own checks leave the value in the file and put a letter in the quality-flag column:

In [7]:
py["qflag"].value_counts(dropna=True).rename("flagged observations").to_frame().assign(
    meaning=lambda d: [QFLAG_MEANING.get(i, "?") for i in d.index])

,flagged observations,meaning
qflag,,
I,626,failed internal consistency check
K,74,failed streak / frequent-value check
S,1,failed spatial consistency check
X,1,failed bounds check


In [8]:
first_flag = py[py.qflag.notna()].head(1)
if first_flag.empty:
    print("no quality-flagged observations at this station")
else:
    d, el = first_flag.date.iloc[0], first_flag.element.iloc[0]
    window = py[(py.element == el) & (py.date.between(d - pd.Timedelta(days=3), d + pd.Timedelta(days=3)))]
    print("a flagged observation, with its neighbours:")
    display(window[["date", "element", "value", "qflag", "sflag"]].reset_index(drop=True))

a flagged observation, with its neighbours:


,date,element,value,qflag,sflag
0,1850-11-24,TMAX,116,NaN,E
1,1850-11-25,TMAX,96,NaN,E
2,1850-11-26,TMAX,74,NaN,E
3,1850-11-27,TMAX,-3,I,E
4,1850-11-28,TMAX,58,NaN,E
5,1850-11-29,TMAX,72,NaN,E
6,1850-11-30,TMAX,78,NaN,E


## Units

Temperatures and precipitation are stored in **tenths** (a TMAX of `380` is 38.0 °C; PRCP `50` is 5.0 mm); snow and snow depth are whole millimetres. Reading the raw integer as degrees is a ten-fold error, so the parser keeps both the raw integer and a converted `value_native`:

In [9]:
sp[sp.element == "TMAX"][["date", "value", "value_native"]].head(3)

,date,value,value_native
2162,1844-01-01,19,1.9
2163,1844-01-02,-2,-0.2
2164,1844-01-03,67,6.7


## What this means for the pipeline

1. **Parse with `stack(31, ...)` in Spark SQL.** One line becomes up to 31 rows with no Python UDF, and the check above shows it matches the plain-Python parse exactly.
2. **Drop `-9999` and padding at parse time.** Completeness is measured later from what remains (notebook 03).
3. **Keep every flag column and never filter flagged rows out.** The value is still the data; whether it is trustworthy is the question notebook 03 asks.
4. **Keep the raw integer and the converted value.** Unit bugs are easy to introduce and easy to check when both are present.

Next: [02 · The Spark pipeline](02_spark_pipeline.ipynb)